In [ ]:
import pandas as pd
from tabulate import tabulate
import matplotlib.pyplot as plt

# === 1. Caricamento e pulizia iniziale ===
file_path = '/Users/pietrocarputo/Desktop/Dataset/CPUUsage.csv'

df = pd.read_csv(file_path)

# Rimuovo colonne con almeno un NaN
cols_with_nan = df.columns[df.isna().any()].tolist()
df = df.drop(columns=cols_with_nan)

if cols_with_nan:
    print("Colonne rimosse per presenza di NaN:")
    for col in cols_with_nan:
        print(f"  - {col}")
else:
    print("Nessuna colonna contenente NaN trovata.")

print(f"Numero di colonne rimanenti: {df.shape[1]}")

# === 2. Conversione timestamp ===
df['Time'] = pd.to_datetime(df['Time'])

# === 3. Suddivisione in segmenti ===
ranges = [
    (0, 721),    # df1
    (722, 1491), # df5
    (1492, 1852),# df6
    (1853, 2213),# df2
    (2214, 2574),# df3
    (2575, 2935),# df4
    (2936, 3118),# df7
    (3119, 3299) # df8
]

grouped_dataframes = []

for i, (start, end) in enumerate(ranges):
    segment = df.iloc[start:end+1].copy()
    segment['Minute'] = segment['Time'].dt.floor('T')
    grouped = segment.groupby('Minute').mean(numeric_only=True).reset_index()

    # Rimuovo colonne che contengono solo zeri (escludendo 'Minute')
    zero_cols = [col for col in grouped.columns if col != 'Minute' and (grouped[col] == 0).all()]
    if zero_cols:
        print(f"DataFrame {i + 1}: colonne rimosse perché contenenti solo zeri:")
        for col in zero_cols:
            print(f"  - {col}")
        grouped = grouped.drop(columns=zero_cols)

    grouped_dataframes.append(grouped)

    print(f"\nDataFrame {i + 1} (righe {start + 1}–{end + 1}):")
    print(f"Lunghezza originale: {len(segment)} righe")
    print(f"Lunghezza raggruppata per minuto: {len(grouped)} righe")
    print("\nPrime 5 righe:")
    print(tabulate(grouped.head(5), headers='keys', tablefmt='pretty'))
    print("\nUltime 5 righe:")
    print(tabulate(grouped.tail(5), headers='keys', tablefmt='pretty'))

# === 4. Calcolo medie e associazione utenti ===
utenti_assoc = [5, 11, 16, 23, 27, 31, 38, 49]  # utenti in ordine crescente

media_totale = []

for i, dfg in enumerate(grouped_dataframes):
    dfg_clean = dfg.dropna(axis=1, how='all')
    numeric_cols = dfg_clean.select_dtypes(include='number').columns
    media_df = dfg_clean[numeric_cols].mean().mean()
    media_totale.append((i, media_df, dfg_clean))

# === 5. Ordina per media crescente e associa utenti crescenti ===
media_totale_sorted = sorted(media_totale, key=lambda x: x[1])
utenti_assoc_ordinati = sorted(utenti_assoc)[:len(media_totale_sorted)]
grouped_dataframes_ordinati = [item[2] for item in media_totale_sorted]

# === 6. Stampa ordinata: media più bassa → meno utenti ===
print("\nDataFrame ordinati per media crescente con utenti associati crescenti:")
for pos, (df_i, media_val, _) in enumerate(media_totale_sorted, 1):
    utenti = utenti_assoc_ordinati[pos - 1]
    print(f"Posizione {pos}: DataFrame {df_i + 1} - Media: {media_val:.4f} - Utenti associati: {utenti}")

# === 7. Grafico coerente con l'ordinamento crescente ===
plt.figure(figsize=(16, 6))
x_offset = 0

for i, (df_i, media_val, df_ord) in enumerate(media_totale_sorted):
    utenti = utenti_assoc_ordinati[i]
    label = f"DataFrame {df_i + 1} ({utenti} utenti)"

    if 'tsdb-mysql-1' in df_ord.columns:
        x_vals = range(x_offset, x_offset + len(df_ord))
        plt.plot(x_vals, df_ord['tsdb-mysql-1'], label=label)
        x_offset += len(df_ord)
    else:
        print(f"Colonna 'tsdb-mysql-1' mancante in DataFrame {df_i + 1}")

plt.title("Valori tsdb-mysql-1 ordinati per media crescente (utenti crescenti)")
plt.xlabel("Indice concatenato")
plt.ylabel("tsdb-mysql-1")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

RIDUZIONE DI 0.35 DEI DATAFRAME 3 5 7 

In [ ]:

# === 5. Riduzioni specifiche in percentuale su DF 3, 5, 7 ===
def reduce_and_clip_percent(df, percent):
    factor = 1 - percent
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols] * factor
    df[numeric_cols] = df[numeric_cols].clip(lower=0)

# Percentuali di riduzione da applicare (35% ciascuno)
reduction_map = {
    3: 0.35,  # df3 → index 2
    5: 0.35,  # df5 → index 4
    7: 0.35   # df7 → index 6
}



for df_num, reduction_percent in reduction_map.items():
    idx = df_num - 1
    reduce_and_clip_percent(grouped_dataframes[idx], reduction_percent)
    print(f"RIDUZIONE APPLICATA a DataFrame {df_num} (-{int(reduction_percent * 100)}%)")

# === 6. Plot: un grafico per ogni colonna, tutti i DF insieme non sovrapposti ===
colonne_numeriche = grouped_dataframes_ordinati[0].select_dtypes(include='number').columns
colonne_numeriche = [col for col in colonne_numeriche if col != 'Minute']

for col in colonne_numeriche:
    plt.figure(figsize=(16, 6))
    x_offset = 0
    for i, df in enumerate(grouped_dataframes_ordinati):
        df_idx = media_totale_sorted[i][0] + 1
        utenti = utenti_assoc_ordinati[i]
        label = f"DF {df_idx} ({utenti} utenti)"

        if col in df.columns:
            y = df[col].values
            x = range(x_offset, x_offset + len(y))
            plt.plot(x, y, label=label)
            x_offset += len(y)
        else:
            print(f"Colonna '{col}' mancante in DF {df_idx}")

    plt.title(f"Andamento '{col}' per tutti i DataFrame (media crescente)")
    plt.xlabel("Indice concatenato")
    plt.ylabel(col)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

CREAZIONE CURVE PER OGNI GIORNO DELLA SETTIMANA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd  # IMPORTANTE: pandas serve per il DataFrame

# === Parametri globali ===
sigma = 0.5  # in ore
minuti_giorno = 1440
giorni_settimana = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom']
utenti_possibili = [5, 11, 16, 23, 27]
utenti_possibili_weekend = [23, 27, 31, 38, 49]
num_settimane = 1
giorni = 7

def genera_probabilita(carico, giorni_settimana, utenti_possibili, utenti_possibili_weekend):
    def distribuzione(carico, utenti):
        n = len(utenti)
        if carico == 'basso':
            pesi = np.linspace(1, 0.2, n)
        elif carico == 'medio':
            pesi = np.ones(n)
        elif carico == 'alto':
            pesi = np.linspace(0.2, 1, n)
        else:
            raise ValueError("Carico deve essere: 'basso', 'medio' o 'alto'")
        return (pesi / pesi.sum()).tolist()

    prob = {}
    for giorno in giorni_settimana:
        if giorno in ['Sab', 'Dom']:
            utenti = utenti_possibili_weekend
        else:
            utenti = utenti_possibili
        prob[giorno] = distribuzione(carico, utenti)
    
    return prob

carico = 'basso'  # oppure 'basso', 'alto'
prob_utenti_per_giorno = genera_probabilita(carico, giorni_settimana, utenti_possibili, utenti_possibili_weekend)

# === Funzioni utili ===
def genera_num_picchi():
    return np.random.randint(1, 5)  # Da 1 a 4 picchi per giorno

def mappa_per_rank(curva, valori_possibili):
    n_classi = len(valori_possibili)
    indici_ordinati = np.argsort(curva)
    output = np.zeros_like(curva)
    taglia_gruppo = len(curva) // n_classi
    for i in range(n_classi):
        start = i * taglia_gruppo
        end = (i + 1) * taglia_gruppo if i < n_classi - 1 else len(curva)
        for idx in indici_ordinati[start:end]:
            output[idx] = valori_possibili[i]
    return output

def genera_slot_variabili(possibili_durata=[30, 45, 60], minuti_target=1440):
    slot = []
    minuti_rimanenti = minuti_target
    while minuti_rimanenti > 0:
        durata_possibili = [d for d in possibili_durata if d <= minuti_rimanenti]
        if not durata_possibili:
            return genera_slot_variabili(possibili_durata, minuti_target)
        durata = np.random.choice(durata_possibili)
        slot.append(durata)
        minuti_rimanenti -= durata
    return slot

def calcola_centro_slot(slot):
    centro_slot = []
    tempo = 0
    for durata in slot:
        centro = tempo + durata / 2
        centro_slot.append(centro)
        tempo += durata
    return np.array(centro_slot)

# Funzione per convertire minuti in stringa HH:MM
def minuti_to_orario(minuti):
    h = minuti // 60
    m = minuti % 60
    return f"{int(h):02d}:{int(m):02d}"

# Posizioni tipiche dei picchi (in minuti)
centro_picchi_base = [8, 13, 18, 22]
centro_picchi_minuti = [x * 60 for x in centro_picchi_base]

# === Simulazione ===
mu = []
slot_durata_settimane = []

for settimana in range(num_settimane):
    mu_settimana = []
    slot_durata_settimana = []

    for d in range(giorni):
        giorno_nome = giorni_settimana[d]
        slot_durata = genera_slot_variabili()
        slot_durata_settimana.append(slot_durata)
        centro_slot = calcola_centro_slot(slot_durata)

        if giorno_nome in prob_utenti_per_giorno:
            if d < 5:
                utenti = utenti_possibili
            else:
                utenti = utenti_possibili_weekend
            prob = prob_utenti_per_giorno[giorno_nome]
            A_d = np.random.choice(utenti, p=prob)
        else:
            utenti = utenti_possibili
            A_d = np.random.choice(utenti)

        num_picchi = genera_num_picchi()
        picchi_d = np.random.normal(
            np.random.choice(centro_picchi_minuti, size=num_picchi, replace=False),
            60  # Deviazione standard in minuti
        )

        print(f"Settimana {settimana} - Giorno {d} ({giorni_settimana[d]}) - Picchi: {picchi_d.round(1)}")

        curve_base = np.array([
            sum(np.exp(-((c - p) ** 2) / (2 * (sigma * 60) ** 2)) for p in picchi_d)
            for c in centro_slot
        ])
        curve_base /= np.max(curve_base)
        quantizzata = mappa_per_rank(curve_base, utenti)

        mu_settimana.append(quantizzata)

    mu.append(mu_settimana)
    slot_durata_settimane.append(slot_durata_settimana)

# === CREAZIONE DATAFRAME ===
rows = []

for settimana in range(num_settimane):
    for d in range(giorni):
        giorno_nome = giorni_settimana[d]
        durata_slot = slot_durata_settimane[settimana][d]
        utenti_slot = mu[settimana][d]

        tempo_inizio = 0
        for i, durata in enumerate(durata_slot):
            inizio = tempo_inizio
            fine = inizio + durata
            num_utenti = utenti_slot[i]

            rows.append({
                'settimana': settimana + 1,
                'giorno': giorno_nome,
                'slot_inizio_minuti': inizio,
                'slot_durata_minuti': durata,
                'slot_fine_minuti': fine,
                'orario_inizio': minuti_to_orario(inizio),
                'orario_fine': minuti_to_orario(fine),
                'num_utenti': num_utenti
            })
            tempo_inizio = fine

df_slot = pd.DataFrame(rows)

# Stampa tutto il DataFrame senza tagli
print(df_slot.to_string(index=False))

# === Visualizzazione: 7 grafici separati per ogni giorno ===
import matplotlib.cm as cm
import matplotlib.ticker as ticker

def format_orario(x, pos):
    h = int(x) // 60
    m = int(x) % 60
    return f"{h:02d}:{m:02d}"

for settimana in range(num_settimane):
    fig, axes = plt.subplots(nrows=7, ncols=1, figsize=(14, 10), sharex=True, sharey=True)

    valori_y = []
    cmap = cm.get_cmap('tab10')

    for d in range(giorni):
        giorno_nome = giorni_settimana[d]
        centro_slot = calcola_centro_slot(slot_durata_settimane[settimana][d])
        utenti = mu[settimana][d]

        # Estendi curva da 00:00 a 24:00
        centro_slot = np.insert(centro_slot, 0, 0)
        utenti = np.insert(utenti, 0, utenti[0])
        centro_slot = np.append(centro_slot, 1440)
        utenti = np.append(utenti, utenti[-1])

        valori_y.append(utenti)
        colore = cmap(d % 10)

        axes[d].plot(centro_slot, utenti, color=colore)
        axes[d].set_ylabel(giorno_nome, rotation=0, labelpad=30, fontsize=10)
        axes[d].grid(True)

        # Asse X in formato HH:MM ogni 2 ore
        axes[d].xaxis.set_major_locator(ticker.MultipleLocator(120))
        axes[d].xaxis.set_major_formatter(ticker.FuncFormatter(format_orario))
        axes[d].set_xlim(0, 1440)

    fig.suptitle(f'Settimana {settimana + 1} - Andamento giornaliero (slot variabili)', fontsize=14)
    axes[-1].set_xlabel('Orario del giorno')
    fig.text(0.04, 0.5, 'Numero utenti', va='center', rotation='vertical')

    all_vals = np.concatenate(valori_y)
    y_min, y_max = np.min(all_vals), np.max(all_vals)
    for ax in axes:
        ax.set_ylim(y_min - 1, y_max + 1)

    plt.tight_layout(rect=[0.05, 0.03, 1, 0.95])
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors

# Setup: prossimo lunedì mezzanotte
oggi = pd.Timestamp.now().normalize()
prossimo_lunedi = oggi + pd.Timedelta((7 - oggi.weekday()) % 7, unit='D')

# Filtra solo lunedì settimana 1
df_lunedi = df_slot[(df_slot['settimana'] == 1) & (df_slot['giorno'] == 'Lun')].copy()
print("\nRiepilogo slot del lunedì prima settimana:")
print(df_lunedi[['slot_inizio_minuti', 'slot_fine_minuti', 'slot_durata_minuti', 'num_utenti']])

# Mappa numero utenti → DataFrame corretto
utente_to_df = dict(zip(utenti_assoc_ordinati, grouped_dataframes_ordinati))

records = []
start_time = prossimo_lunedi

for i, row in df_lunedi.iterrows():
    durata = int(row['slot_durata_minuti'])
    inizio_slot = start_time + pd.Timedelta(minutes=int(row['slot_inizio_minuti']))
    fine_slot = start_time + pd.Timedelta(minutes=int(row['slot_fine_minuti']))
    num_utenti = int(row['num_utenti'])

    df_corrente = utente_to_df[num_utenti].copy()
    if 'Minute' in df_corrente.columns:
        df_corrente = df_corrente.drop(columns=['Minute'])

    ridotto_idx = i % len(df_corrente)
    base_values = df_corrente.iloc[ridotto_idx]

    print(f"\nSlot {i}: {inizio_slot.strftime('%H:%M')} - {fine_slot.strftime('%H:%M')} | Durata: {durata} min | Utenti: {num_utenti}")

    for minuto in range(durata):
        timestamp = inizio_slot + pd.Timedelta(minutes=minuto)
        noisy_values = base_values * (1 + np.random.normal(0, 0.05, size=base_values.shape))
        noisy_values = np.clip(noisy_values, 0, None)

        record = {'Timestamp': timestamp}
        record.update(noisy_values.to_dict())
        records.append(record)

df_expanded = pd.DataFrame(records)

# === Visualizzazione ===
min_utenti = df_lunedi['num_utenti'].min()
max_utenti = df_lunedi['num_utenti'].max()
norm = mcolors.Normalize(vmin=min_utenti, vmax=max_utenti)
cmap = plt.cm.get_cmap('YlOrRd')

cols = [c for c in df_expanded.columns if c != 'Timestamp']

for col in cols:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_expanded['Timestamp'], df_expanded[col], label=col, color='blue')
    ax.set_title(f"Andamento lunedì - Colonna: {col}")
    ax.set_xlabel('Ora')
    ax.set_ylabel('Valore (con rumore 5%)')
    ax.grid(True)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    fig.autofmt_xdate()

    for _, slot in df_lunedi.iterrows():
        start = start_time + pd.Timedelta(minutes=slot['slot_inizio_minuti'])
        end = start_time + pd.Timedelta(minutes=slot['slot_fine_minuti'])
        color = cmap(norm(slot['num_utenti']))
        ax.axvspan(start, end, color=color, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [5]:
prima_colonna = df_expanded.columns[1]  # Prende la prima colonna dopo 'Timestamp'
print(f"\nDataFrame della prima colonna '{prima_colonna}' del lunedì:")
print(df_expanded[['Timestamp', prima_colonna]])


DataFrame della prima colonna 'nacosdb-mysql-0' del lunedì:
               Timestamp  nacosdb-mysql-0
0    2025-05-26 00:00:00         1.240487
1    2025-05-26 00:01:00         1.230394
2    2025-05-26 00:02:00         1.214045
3    2025-05-26 00:03:00         1.227795
4    2025-05-26 00:04:00         1.285684
...                  ...              ...
1435 2025-05-26 23:55:00         1.382321
1436 2025-05-26 23:56:00         1.302742
1437 2025-05-26 23:57:00         1.402901
1438 2025-05-26 23:58:00         1.359107
1439 2025-05-26 23:59:00         1.298956

[1440 rows x 2 columns]


1 SETTIMANA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors

# Setup: prossimo lunedì mezzanotte
oggi = pd.Timestamp.now().normalize()
prossimo_lunedi = oggi + pd.Timedelta((7 - oggi.weekday()) % 7, unit='D')

# Filtra tutti i giorni della prima settimana (settimana=1)
df_settimana1 = df_slot[df_slot['settimana'] == 1].copy()

# Mappa numero utenti → DataFrame corretto
utente_to_df = dict(zip(utenti_assoc_ordinati, grouped_dataframes_ordinati))

records = []
start_time = prossimo_lunedi

for i, row in df_settimana1.iterrows():
    durata = int(row['slot_durata_minuti'])
    # Calcolo offset in giorni per il giorno della settimana
    giorno_offset = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom'].index(row['giorno'])
    inizio_slot = start_time + pd.Timedelta(days=giorno_offset, minutes=int(row['slot_inizio_minuti']))
    fine_slot = start_time + pd.Timedelta(days=giorno_offset, minutes=int(row['slot_fine_minuti']))
    num_utenti = int(row['num_utenti'])

    df_corrente = utente_to_df[num_utenti].copy()
    if 'Minute' in df_corrente.columns:
        df_corrente = df_corrente.drop(columns=['Minute'])

    ridotto_idx = i % len(df_corrente)
    base_values = df_corrente.iloc[ridotto_idx]

    print(f"\nSlot {i}: {inizio_slot.strftime('%a %H:%M')} - {fine_slot.strftime('%a %H:%M')} | Durata: {durata} min | Utenti: {num_utenti}")

    for minuto in range(durata):
        timestamp = inizio_slot + pd.Timedelta(minutes=minuto)
        noisy_values = base_values * (1 + np.random.normal(0, 0.05, size=base_values.shape))
        noisy_values = np.clip(noisy_values, 0, 100)

        record = {'Timestamp': timestamp}
        record.update(noisy_values.to_dict())
        records.append(record)

df_expanded = pd.DataFrame(records)

# Normalizzazione per la colormap (min/max utenti)
min_utenti = df_settimana1['num_utenti'].min()
max_utenti = df_settimana1['num_utenti'].max()
norm = mcolors.Normalize(vmin=min_utenti, vmax=max_utenti)
cmap = plt.cm.get_cmap('YlOrRd')

cols = [c for c in df_expanded.columns if c != 'Timestamp']

# Grafico per ogni colonna
for col in cols:
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(df_expanded['Timestamp'], df_expanded[col], label=col, color='blue')
    ax.set_title(f"Andamento prima settimana - Colonna: {col}")
    ax.set_xlabel('Giorno e ora')
    ax.set_ylabel('Valore (con rumore 5%)')
    ax.grid(True)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %H:%M'))
    fig.autofmt_xdate()

    # Colorazione sfondo per ogni slot in tutta la settimana
    for _, slot in df_settimana1.iterrows():
        giorno_offset = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom'].index(slot['giorno'])
        start = start_time + pd.Timedelta(days=giorno_offset, minutes=slot['slot_inizio_minuti'])
        end = start_time + pd.Timedelta(days=giorno_offset, minutes=slot['slot_fine_minuti'])
        color = cmap(norm(slot['num_utenti']))
        ax.axvspan(start, end, color=color, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
records = []

# Ciclo su 5 settimane (0 = prima settimana)
for settimana_offset in range(5):
    week_start = start_time + pd.Timedelta(weeks=settimana_offset)

    for i, row in df_settimana1.iterrows():
        durata = int(row['slot_durata_minuti'])
        giorno_offset = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom'].index(row['giorno'])
        inizio_slot = week_start + pd.Timedelta(days=giorno_offset, minutes=int(row['slot_inizio_minuti']))
        fine_slot = week_start + pd.Timedelta(days=giorno_offset, minutes=int(row['slot_fine_minuti']))
        num_utenti = int(row['num_utenti'])

        df_corrente = utente_to_df[num_utenti].copy()
        if 'Minute' in df_corrente.columns:
            df_corrente = df_corrente.drop(columns=['Minute'])

        ridotto_idx = i % len(df_corrente)
        base_values = df_corrente.iloc[ridotto_idx]

        print(f"Settimana {settimana_offset+1}, Slot {i}: {inizio_slot.strftime('%a %H:%M')} - {fine_slot.strftime('%a %H:%M')} | Durata: {durata} min | Utenti: {num_utenti}")

        for minuto in range(durata):
            timestamp = inizio_slot + pd.Timedelta(minutes=minuto)

            # Rumore tra 5% e 10%
            noise_percent = np.random.uniform(0.05, 0.10)
            noisy_values = base_values * (1 + np.random.normal(0, noise_percent, size=base_values.shape))
            noisy_values = np.clip(noisy_values, 0, 100)

            record = {'Timestamp': timestamp}
            record.update(noisy_values.to_dict())
            records.append(record)

df_expanded = pd.DataFrame(records)

In [ ]:
# Normalizzazione per la colormap (min/max utenti)
min_utenti = df_settimana1['num_utenti'].min()
max_utenti = df_settimana1['num_utenti'].max()
norm = mcolors.Normalize(vmin=min_utenti, vmax=max_utenti)
cmap = plt.cm.get_cmap('YlOrRd')

cols = [c for c in df_expanded.columns if c != 'Timestamp']

# Grafico per ogni colonna
for col in cols:
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(df_expanded['Timestamp'], df_expanded[col], label=col, color='blue')
    ax.set_title(f"Andamento su 5 settimane - Colonna: {col}")
    ax.set_xlabel('Giorno e ora')
    ax.set_ylabel('Valore (con rumore 5–10%)')
    ax.grid(True)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d-%m %H:%M'))
    fig.autofmt_xdate()

    # Sfondo colorato per ogni slot, in ogni settimana
    for settimana_offset in range(5):
        week_start = start_time + pd.Timedelta(weeks=settimana_offset)
        for _, slot in df_settimana1.iterrows():
            giorno_offset = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom'].index(slot['giorno'])
            start = week_start + pd.Timedelta(days=giorno_offset, minutes=slot['slot_inizio_minuti'])
            end = week_start + pd.Timedelta(days=giorno_offset, minutes=slot['slot_fine_minuti'])
            color = cmap(norm(slot['num_utenti']))
            ax.axvspan(start, end, color=color, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [9]:
# Rimuovi le ultime 3 colonne
df_to_save = df_expanded.iloc[:, :-3]

# Salva in un file CSV
df_to_save.to_csv('dati_settimane.csv', index=False)